# K-fold data prep

In [ ]:
from sklearn.model_selection import KFold

import pandas as pd
import numpy as np
import datasets

import warnings
warnings.simplefilter(action='ignore')

## Loading data

In [ ]:
df = pd.read_parquet("hf://datasets/malmaud/onestop_qa/data/train-00000-of-00001.parquet")

indices = np.arange(len(df)//3//3)

# Sort the dataset by levels and separate
df = df.sort_values('level',kind='stable').reset_index(drop=True)
l1_df = df.loc[0:485]
l2_df = df.loc[486:486+485].reset_index(drop=True)
l3_df = df.loc[486+485+1:486+485+1+485].reset_index(drop=True)

In [ ]:
kf = KFold(n_splits=9,random_state=42,shuffle=True)
kf.get_n_splits(indices)

import itertools
def pairwise(iterable):
    "s -> (s0, s1), (s1, s2), (s2, s3), ..., (s8, s0)"
    a, b = itertools.tee(iterable)
    next(b, None)
    return zip(a, b)

test_indices = []
for i, (train_index, test_index) in enumerate(kf.split(indices)):
    test_indices.append(test_index)

test_val_indices = list(pairwise(test_indices))
test_val_indices.append((test_indices[-1],test_indices[0]))

In [ ]:
## Specifying B-type data

test_dfs = []
for i, test_idx in enumerate(test_indices):
    # Use only level 1 version of the passages for testing
    q_indices_test = sorted(np.concatenate((test_idx * 3, test_idx * 3 + 1, test_idx * 3 + 2)))
    test_df = l1_df.iloc[q_indices_test,:]
    test_df['correct'] = test_df['answers'].str[0]
    test_df = test_df.explode('answers').reset_index(drop=False)
    test_df['paragraph'] = test_df['paragraph'].str.split()
    test_df['option'] = ['A', 'B', 'C', 'D'] * len(q_indices_test)
    test_df['qa'] = test_df['question'] + ' ' + test_df['answers']
    test_df['qa'] = test_df['qa'].str.split()
    test_df = test_df.drop(axis='index', index=[item for item in test_df.index.to_list() if item % 4 == 0])
    
    # Select only B
    test_df_B = test_df[test_df['option'] == 'B'].reset_index(drop=True)
    test_dfs.append(test_df_B)

## Testing functions (ignore this)

In [ ]:
test_df_B = test_dfs[7]

a_spans = []
articles = []
for i, item in test_df_B.loc[0:].iterrows():
    q_id = item['index']
    article = ' '.join(item['paragraph'])
    question = item['question']
    answer = item['correct']
    distractor = item['answers']
    a_span = ''

    for a_start, a_end in zip(item['a_span'][0::2], item['a_span'][1::2]):
        a_span += ' '.join(item['paragraph'][a_start:a_end+1])

    a_spans.append(a_span)
    articles.append(article)

test_df_B['article'] = articles
test_df_B['a_span'] = a_spans

test_df_B.head(3)

## Preparing all data

In [ ]:
## Update the prompt if you need to create the training dataset for other types

def create_prompts(df):
    instructions = []
    outputs = []
    for i, item in df.loc[0:].iterrows():

        q_id = item['index']
        article = ' '.join(item['paragraph'])
        question = item['question']
        answer = item['correct']
        distractor = item['answers']
        a_span = ''
        d_span = ''

        for a_start, a_end in zip(item['a_span'][0::2], item['a_span'][1::2]): # iterate every two elements
            a_span += ' '.join(item['paragraph'][a_start:a_end+1])

        for d_start, d_end in zip(item['d_span'][0::2], item['d_span'][1::2]): # iterate every two elements
            d_span += ' '.join(item['paragraph'][d_start:d_end+1])

        prompt = f"""
        The following passage, question, and answer are from a multiple-choice reading comprehension task:

        Passage: {article}
        Question: {question}
        Answer: {answer}

        The following is the critical span from the passage on which the correct answer is based:
        Critical Span: {a_span}

        Please create one distractor based on the given passage, question, answer, and the critical span.

        Requirements:
        1. The distractor is based on the information in the critical span, and does not include new information not given in the span.
        2. The distractor is similar in length to the given answer.
        3. The distractor should be a plausible option to the given multiple-choice question.
        4. The distractor must be incorrect; it does not constitute an acceptable correct answer.

        First, repeat the critical span. Then, respond with the distractor created based on the critical span and nothing else.
        """

        output = f"""
        Critical Span: {a_span}
        Distractor: {distractor}
        """
        instructions.append(prompt.replace("\n        ",'\n').lstrip().rstrip())
        outputs.append(output.replace("\n        ",'\n').lstrip().rstrip())
    return instructions, outputs

In [ ]:
## Creating the k-fold training data
## For B type, if for other types, please modify the function above first

output_path = 'k-fold/'

for i, (test_idx, val_idx) in enumerate(test_val_indices):
    train_idx = np.setdiff1d(indices, np.append(test_idx,val_idx))

    # Use only level 1 version of the passages for testing
    q_indices_test = sorted(np.concatenate((test_idx * 3, test_idx * 3 + 1, test_idx * 3 + 2)))
    test_df = l1_df.iloc[q_indices_test,:]
    test_df['correct'] = test_df['answers'].str[0]
    test_df = test_df.explode('answers').reset_index(drop=False)
    test_df['paragraph'] = test_df['paragraph'].str.split()
    test_df['option'] = ['A', 'B', 'C', 'D'] * len(q_indices_test)
    test_df['qa'] = test_df['question'] + ' ' + test_df['answers']
    test_df['qa'] = test_df['qa'].str.split()
    test_df = test_df.drop(axis='index', index=[item for item in test_df.index.to_list() if item % 4 == 0])
    # testset = Dataset.from_pandas(test_df)

    # valdf
    q_indices_val = sorted(np.concatenate((val_idx * 3, val_idx * 3 + 1, val_idx * 3 + 2)))
    val_df = l1_df.iloc[q_indices_val,:]
    val_df['correct'] = val_df['answers'].str[0]
    val_df = val_df.explode('answers').reset_index(drop=False)
    val_df['paragraph'] = val_df['paragraph'].str.split()
    val_df['option'] = ['A', 'B', 'C', 'D'] * len(q_indices_val)
    val_df['qa'] = val_df['question'] + ' ' + val_df['answers']
    val_df['qa'] = val_df['qa'].str.split()
    val_df = val_df.drop(axis='index', index=[item for item in val_df.index.to_list() if item % 4 == 0])
    # valset = Dataset.from_pandas(val_df)

    # Use passages in all three levels as if this is data augmentation
    q_indices_train = sorted(np.concatenate((train_idx * 3, train_idx * 3 + 1, train_idx * 3 + 2))) # three q's per passage
    train_df = pd.concat((l1_df.iloc[q_indices_train,:],l2_df.iloc[q_indices_train,:],l3_df.iloc[q_indices_train,:]))
    train_df['correct'] = train_df['answers'].str[0]
    train_df = train_df.explode('answers').reset_index(drop=False)
    train_df['paragraph'] = train_df['paragraph'].str.split()
    train_df['option'] = ['A', 'B', 'C', 'D'] * len(q_indices_train) * 3
    train_df['qa'] = train_df['question'] + ' ' + train_df['answers']
    train_df['qa'] = train_df['qa'].str.split()
    train_df = train_df.drop(axis='index', index=[item for item in train_df.index.to_list() if item % 4 == 0])
    # trainset = Dataset.from_pandas(train_df)

    ## For B type, if for other types, please modify the function above first
    ## And then modify the variables, option etc. below
    train_df_B = train_df[train_df['option'] == 'B'].reset_index(drop=True)
    val_df_B = val_df[val_df['option'] == 'B'].reset_index(drop=True)
    test_df_B = test_df[test_df['option'] == 'B'].reset_index(drop=True)

    train_instructions, train_outputs = create_prompts(train_df_B)
    val_instructions, val_outputs = create_prompts(val_df_B)
    test_instructions, test_outputs = create_prompts(test_df_B)

    train_df_B["instruction"] = train_instructions
    train_df_B["output"] = train_outputs
    train_df_B["input"] = ["" for _ in range(len(train_outputs))]

    val_df_B["instruction"] = val_instructions
    val_df_B["output"] = val_outputs
    val_df_B["input"] = ["" for _ in range(len(val_outputs))]

    test_df_B["instruction"] = test_instructions
    test_df_B["output"] = test_outputs
    test_df_B["input"] = ["" for _ in range(len(test_outputs))]

    test_df_B[['instruction','input','output']].to_json(f"{output_path}test/instruct_test_fold_{i+1}.json", orient='records', lines=True, force_ascii=False)
    train_df_B[['instruction','input','output']].to_json(f"{output_path}train/instruct_train_fold_{i+1}.json", orient='records', lines=True, force_ascii=False)
    val_df_B[['instruction','input','output']].to_json(f"{output_path}val/instruct_val_fold_{i+1}.json", orient='records', lines=True, force_ascii=False)

## Downloading pre-trained model

In [ ]:
# !pip install -U "huggingface_hub[cli]"
# !huggingface-cli login --token $YOUR-TOKEN --add-to-git-credential
# !huggingface-cli download meta-llama/Llama-3.1-8B --local-dir F:\Llama-3.1-8B

# This may take some time